# Latent-displacement embeddings in 2D

Load one persisted latent-displacement model for every configured dataset and inspect the full MIMIC embedding, recolored by each original feature.

In [ ]:
EXPERIMENT_NAME = "latent-displacement-best__mode-factorised__capacity-0.85__neighbors-14__lambda-0.25-1.1__rows-4000"
N_COLUMNS = 4
ARTIFACT_DIR = None

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT / "notebooks", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from mimic_notebook_utils import plot_feature_embedding_grid, plot_global_embedding_color_grid
from streamlined.config import experiment_artifact_dir
from streamlined.datasets import feature_display_names
from streamlined.embedding_visualization import (
    load_latest_latent_displacement_run,
    load_latent_displacement_embedding,
)

In [ ]:
artifact_base_dir = ARTIFACT_DIR or EXPERIMENT_ROOT / "artifacts"
artifact_dir = experiment_artifact_dir(artifact_base_dir, EXPERIMENT_NAME)
latest_run = load_latest_latent_displacement_run(artifact_dir)
config = latest_run.config
SAMPLE_SIZE = latest_run.sample_size
SEED = latest_run.seed

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Datasets: {', '.join(config.profile.datasets)}")
print(f"Latent-displacement models: n={SAMPLE_SIZE}, seed={SEED}")

## Full and per-feature embeddings colored by each feature

For each dataset, the first grid uses one shared 2D projection of the persisted full embedding and recolors it by every raw feature. The second grid projects each feature-specific embedding block separately and colors it by that same feature. The fitted models, plotting frames, and run configuration are loaded from the latest successful execution of `03_train_latent_displacement_models.ipynb`; this notebook does not refit them.

In [ ]:
for dataset_key in config.profile.datasets:
    print("-" * 120)
    print(f"{dataset_key}: loading latent-displacement embedding")
    view = load_latent_displacement_embedding(
        config,
        dataset_key=dataset_key,
        sample_size=SAMPLE_SIZE,
        seed=SEED,
    )
    fig, axes = plot_global_embedding_color_grid(
        view.model,
        view.plot_frame,
        columns=view.feature_columns,
        display_names=feature_display_names(dataset_key, view.feature_columns),
        n_cols=N_COLUMNS,
        size=(4, 4),
        random_state=SEED,
    )
    fig.suptitle(
        f"{dataset_key}: latent displacement full embedding (fitted n={view.fitted_rows})",
        y=1.01,
    )
    display(fig)
    plt.close(fig)

    feature_fig, feature_axes = plot_feature_embedding_grid(
        view.model,
        view.plot_frame,
        columns=view.feature_columns,
        display_names=feature_display_names(dataset_key, view.feature_columns),
        n_cols=N_COLUMNS,
        size=(4, 4),
        random_state=SEED,
    )
    feature_fig.suptitle(
        f"{dataset_key}: per-feature embeddings colored by their own values",
        y=1.01,
    )
    display(feature_fig)
    plt.close(feature_fig)